# DeepGuard — Video Preprocessing Pipeline (Updated & Fixed)

This notebook prepares FaceForensics++ C23 videos for DeepGuard deepfake classification.

**Pipeline Architecture:**
FaceForensics++ C23 → Video-level Stratified Split → Frame Sampling → MTCNN Face Detection → Largest Face Crop (224x224) → Save Frames & Verified Balanced Metadata.

In [ ]:
# Install dependencies
!pip uninstall -y torchcodec
!pip install -q -U "datasets>=3.0.0" huggingface_hub scikit-learn "pandas==2.2.3" matplotlib seaborn tqdm requests opencv-python fsspec
!pip install -q --no-build-isolation --no-deps facenet-pytorch

Found existing installation: torchcodec 0.11.0+cu128
Uninstalling torchcodec-0.11.0+cu128:
  Successfully uninstalled torchcodec-0.11.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 12.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the 

In [ ]:
# Environment Imports and Setup
import os
import re
import json
import random
import hashlib
import tempfile
import warnings
from pathlib import Path

import cv2
import fsspec
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torchvision
from datasets import load_dataset
from facenet_pytorch import MTCNN
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from google.colab import drive

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | Device: {device}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

drive.mount("/content/drive")

PyTorch: 2.11.0+cu128 | Device: cuda
GPU: Tesla T4
Mounted at /content/drive


In [ ]:
# Global Configuration & Directories
DATASET_ID = "bitmind/FaceForensicsC23"
PILOT_MODE = True

# Split ratios
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

# Pilot sizes (Guarantees class balance in Pilot mode)
PILOT_TRAIN_REAL, PILOT_TRAIN_FAKE = 100, 100
PILOT_VAL_REAL, PILOT_VAL_FAKE = 25, 25
PILOT_TEST_REAL, PILOT_TEST_FAKE = 25, 25

# Face extraction configuration
FRAMES_PER_VIDEO = 10
IMAGE_SIZE = 224
MIN_FACE_PROB = 0.90
FACE_PADDING = 0.15
RANDOM_SEED = 42

# Reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# Setup directories
PROJECT_DIR = Path("/content/drive/MyDrive/DeepGuard")
DATA_DIR = PROJECT_DIR / "dataset_analysis"
METADATA_DIR = DATA_DIR / "metadata"
PROCESSED_DIR = DATA_DIR / "processed_faces"
SPLIT_DIR = DATA_DIR / "splits"
LOG_DIR = DATA_DIR / "preprocessing_logs"

for directory in [DATA_DIR, METADATA_DIR, PROCESSED_DIR, SPLIT_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

for split in ["train", "val", "test"]:
    for label_name in ["real", "fake"]:
        (PROCESSED_DIR / split / label_name).mkdir(parents=True, exist_ok=True)

print("Project structure setup complete.")

Project structure setup complete.


In [ ]:
# Load and Standardize Metadata with Robust Label Detection
METADATA_FILE = METADATA_DIR / "ffpp_c23_full_metadata.csv"
if not METADATA_FILE.exists():
    raise FileNotFoundError(f"{METADATA_FILE} not found. Ensure exploration step is complete.")

metadata = pd.read_csv(METADATA_FILE)

def find_column(df, candidates):
    for name in candidates:
        if name in df.columns:
            return name
    return None

path_col = find_column(metadata, ["relative_path", "video_path", "path", "filename"])
label_col = find_column(metadata, ["label", "Label", "target"])
method_col = find_column(metadata, ["method", "category", "manipulation"])

metadata["video_path"] = metadata[path_col].astype(str)
metadata["method"] = metadata[method_col].astype(str) if method_col else "Unknown"

def normalize_label(row):
    # Check explicit label column first
    if label_col and pd.notna(row[label_col]):
        val = str(row[label_col]).strip().lower()
        if val in {"real", "1", "1.0", "true", "original"}:
            return 1
        if val in {"fake", "0", "0.0", "false", "manipulated"}:
            return 0

    # Fallback to path checking
    p = str(row["video_path"]).lower()
    if "original" in p or "real" in p or "actors" in p or "youtube" in p:
        return 1
    return 0

metadata["label"] = metadata.apply(normalize_label, axis=1)
metadata["label_name"] = metadata["label"].map({0: "FAKE", 1: "REAL"})

def clean_video_id(path):
    p = str(path).replace("zip://", "").split("::")[0]
    filename = Path(p).stem
    path_hash = hashlib.md5(p.encode("utf-8")).hexdigest()[:8]
    return f"{filename}_{path_hash}"

metadata["video_id"] = metadata["video_path"].apply(clean_video_id)

# Ensure clean deduplication by video_id
metadata = metadata.drop_duplicates(subset=["video_id"]).reset_index(drop=True)
print(f"Metadata loaded successfully: {len(metadata)} entries.")
print("Class balance in raw metadata:")
print(metadata["label_name"].value_counts())

Metadata loaded successfully: 7000 entries.
Class balance in raw metadata:
label_name
FAKE    6000
REAL    1000
Name: count, dtype: int64


In [ ]:
# Create Video-Level Dataset Splits with Group Stratification
train_df, temp_df = train_test_split(
    metadata,
    test_size=VAL_RATIO + TEST_RATIO,
    stratify=metadata["label"],
    random_state=RANDOM_SEED
)

relative_test_size = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
val_df, test_df = train_test_split(
    temp_df,
    test_size=relative_test_size,
    stratify=temp_df["label"],
    random_state=RANDOM_SEED
)

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

all_splits = pd.concat([train_df, val_df, test_df], ignore_index=True)
all_splits.to_csv(SPLIT_DIR / "ffpp_c23_video_splits.csv", index=False)

def balanced_sample(df, real_count, fake_count, seed=RANDOM_SEED):
    real = df[df["label"] == 1]
    fake = df[df["label"] == 0]
    r_cnt = min(real_count, len(real))
    f_cnt = min(fake_count, len(fake))
    return pd.concat([
        real.sample(r_cnt, random_state=seed),
        fake.sample(f_cnt, random_state=seed)
    ], ignore_index=True)

if PILOT_MODE:
    train_work = balanced_sample(train_df, PILOT_TRAIN_REAL, PILOT_TRAIN_FAKE)
    val_work = balanced_sample(val_df, PILOT_VAL_REAL, PILOT_VAL_FAKE)
    test_work = balanced_sample(test_df, PILOT_TEST_REAL, PILOT_TEST_FAKE)
else:
    train_work, val_work, test_work = train_df.copy(), val_df.copy(), test_df.copy()

work_manifest = pd.concat([train_work, val_work, test_work], ignore_index=True)
MANIFEST_FILE = SPLIT_DIR / "preprocessing_manifest.csv"
work_manifest.to_csv(MANIFEST_FILE, index=False)

print(f"Preprocessing manifest created with {len(work_manifest)} items.")
print("Manifest split breakdown:")
print(pd.crosstab(work_manifest["split"], work_manifest["label_name"]))

Preprocessing manifest created with 300 items.
Manifest split breakdown:
label_name  FAKE  REAL
split                 
test          25    25
train        100   100
val           25    25


In [ ]:
# Detection Model and Preprocessing Utility Functions
mtcnn = MTCNN(
    image_size=IMAGE_SIZE,
    margin=0,
    min_face_size=40,
    thresholds=[0.6, 0.7, 0.7],
    factor=0.709,
    post_process=False,
    keep_all=True,
    device=device
)

def normalize_video_path(path):
    if not path:
        return ""
    p = str(path).replace("zip://", "").split("::")[0]
    p = p.replace("\\", "/").strip("/")
    return Path(p).name  # Match on standard filename for reliable matching

def get_sample_indices(num_frames, frames_per_video):
    if num_frames is None or int(num_frames) <= 0:
        return []
    num_frames = int(num_frames)
    if num_frames <= frames_per_video:
        return list(range(num_frames))
    return np.linspace(0, num_frames - 1, frames_per_video, dtype=int).tolist()

def crop_largest_face(frame_rgb, boxes, probs, min_prob=MIN_FACE_PROB, padding=FACE_PADDING):
    if boxes is None or probs is None:
        return None
    candidates = []
    for box, prob in zip(boxes, probs):
        if prob is None or float(prob) < min_prob:
            continue
        x1, y1, x2, y2 = map(float, box)
        w, h = x2 - x1, y2 - y1
        if w > 0 and h > 0:
            candidates.append({"box": (x1, y1, x2, y2), "prob": float(prob), "area": float(w * h)})

    if not candidates:
        return None

    selected = max(candidates, key=lambda x: x["area"])
    x1, y1, x2, y2 = selected["box"]
    pad_x, pad_y = (x2 - x1) * padding, (y2 - y1) * padding

    x1 = max(0, int(x1 - pad_x))
    y1 = max(0, int(y1 - pad_y))
    x2 = min(frame_rgb.shape[1], int(x2 + pad_x))
    y2 = min(frame_rgb.shape[0], int(y2 + pad_y))

    crop = frame_rgb[y1:y2, x1:x2]
    if crop.size == 0:
        return None

    image = Image.fromarray(crop).resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
    return {
        "image": image,
        "confidence": selected["prob"],
        "area": selected["area"],
        "face_count": len(candidates)
    }

print("MTCNN initialized and support functions defined.")

MTCNN initialized and support functions defined.


In [ ]:
# Streaming Pipeline & Face Extraction Step (With Auto-Checkpointing & Resumption)
META_SAVE_PATH = LOG_DIR / "processed_faces_metadata.csv"

# 1. Load Existing Checkpoint from Google Drive (if available)
if META_SAVE_PATH.exists():
    try:
        existing_df = pd.read_csv(META_SAVE_PATH)
        completed_video_ids = set(existing_df["video_id"].unique())
        extracted_faces_meta = existing_df.to_dict("records")
        print(f"🔄 Checkpoint found! Resuming run. Already processed: {len(completed_video_ids)} videos.")
    except Exception as e:
        print(f"⚠️ Checkpoint read warning: {e}. Starting fresh.")
        completed_video_ids = set()
        extracted_faces_meta = []
else:
    completed_video_ids = set()
    extracted_faces_meta = []

# Prepare manifest lookups
manifest_lookup = {
    normalize_video_path(row["video_path"]): row.to_dict()
    for _, row in work_manifest.iterrows()
}
target_paths = set(manifest_lookup.keys())

# Remove targets that were ALREADY processed in previous runs
already_done_targets = {
    target for target, meta in manifest_lookup.items()
    if meta["video_id"] in completed_video_ids
}
target_paths -= already_done_targets

print(f"🎯 Target videos remaining to process: {len(target_paths)} / {len(work_manifest)}")

# 2. Load streaming dataset
ds = load_dataset(DATASET_ID, split="train", streaming=True).decode(False)

pbar = tqdm(total=len(work_manifest), desc="Processing Target Videos")
pbar.update(len(completed_video_ids))  # Fast-forward progress bar to current checkpoint count

for sample in ds:
    if len(target_paths) == 0:
        break

    v_data = sample.get("video")
    if isinstance(v_data, dict):
        raw_path = v_data.get("path") or v_data.get("filename") or ""
    else:
        raw_path = str(v_data)

    norm_filename = normalize_video_path(raw_path)
    matched_target = norm_filename if norm_filename in target_paths else None

    if not matched_target:
        continue

    video_meta = manifest_lookup[matched_target]
    v_id = video_meta["video_id"]

    # 3. Double-check skip safety net
    if v_id in completed_video_ids:
        target_paths.remove(matched_target)
        pbar.update(1)
        continue

    split_name = video_meta["split"]
    label_dir = video_meta["label_name"].lower()

    # Write bytes out temporarily to stream via OpenCV
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tmp:
        tmp_path = tmp.name
        if isinstance(v_data, dict) and "bytes" in v_data and v_data["bytes"]:
            tmp.write(v_data["bytes"])
        elif isinstance(v_data, dict) and "path" in v_data:
            with fsspec.open(v_data["path"], "rb") as f:
                tmp.write(f.read())

    cap = cv2.VideoCapture(tmp_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    sample_indices = get_sample_indices(total_frames, FRAMES_PER_VIDEO)
    target_indices_set = set(sample_indices)

    sampled_frames = []
    frame_indices = []

    current_idx = 0
    while cap.isOpened() and len(sampled_frames) < len(sample_indices):
        ret, frame = cap.read()
        if not ret:
            break
        if current_idx in target_indices_set:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            sampled_frames.append(frame_rgb)
            frame_indices.append(current_idx)
        current_idx += 1

    cap.release()
    try:
        os.remove(tmp_path)
    except Exception:
        pass

    # Batched MTCNN Detection on GPU
    if sampled_frames:
        boxes_batch, probs_batch = mtcnn.detect(sampled_frames)

        for frame_rgb, idx, boxes, probs in zip(sampled_frames, frame_indices, boxes_batch, probs_batch):
            crop_res = crop_largest_face(frame_rgb, boxes, probs)

            if crop_res:
                img_filename = f"{v_id}_frame_{idx:05d}.jpg"
                save_path = PROCESSED_DIR / split_name / label_dir / img_filename
                crop_res["image"].save(save_path, quality=90)

                extracted_faces_meta.append({
                    "image_path": str(save_path),
                    "video_id": v_id,
                    "frame_index": idx,
                    "split": split_name,
                    "label": video_meta["label"],
                    "label_name": video_meta["label_name"],
                    "confidence": crop_res["confidence"],
                    "face_count": crop_res["face_count"]
                })

    # Update completion sets
    completed_video_ids.add(v_id)
    target_paths.remove(matched_target)
    pbar.update(1)

    # 4. Incremental Checkpoint Save to Google Drive after EACH video
    if extracted_faces_meta:
        pd.DataFrame(extracted_faces_meta).to_csv(META_SAVE_PATH, index=False)

pbar.close()

# Final Verification
faces_df = pd.DataFrame(extracted_faces_meta)
print(f"\n✅ Extraction finished. Total faces saved: {len(faces_df)}")

if not faces_df.empty:
    crosstab = pd.crosstab(faces_df["split"], faces_df["label_name"])
    print("\n=== Processed Faces Split Balance ===")
    print(crosstab)
    assert len(faces_df["label_name"].unique()) == 2, "CRITICAL ERROR: Metadata contains only ONE class!"
    print("\nPipeline successfully generated a balanced dataset for model training!")

README.md:   0%|          | 0.00/460 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


Processing Target Videos:   0%|          | 0/294 [00:00<?, ?it/s]

Extraction finished. Total faces saved: 2929
=== Processed Faces Split Balance ===
label_name  FAKE  REAL
split                 
test         249   250
train        932  1000
val          248   250
Pipeline successfully generated a balanced dataset for model training!
